In [ ]:
# OG Build Model
from docplex.cp.model import CpoModel

from collections import defaultdict

from datagen import generate_data
from instance import Instance

# from docplex.cp.solver.solver_listener import CpoSolverListener


def build_model(instance, *args, **kwargs):
    """
    Intervals per (s,k,e):
      - F[s,k,e] : fetch (size = rt[k])
      - P[s,k,e] : pick  (size = p[k])
      - R[s,k,e] : return(size = rt_return[k])
      - B[s,k,e] : bin presence at station (free size) with start(B)=end(F), end(B)=start(R)

    Station capacity: no_overlap over B[s,*,*] (exactly one bin at station).
    Global single-bin per SKU: no_overlap over {F,B,R} across all stations.
    """
    if not isinstance(instance, Instance):
        S = instance
        L = args[0]
        K = args[1]
        orders_req = args[2]
        rt = args[3]
        p = args[4]
        rt_return = kwargs.get("rt_return", args[5] if len(args) > 5 else None)
        add_symmetry_breaking = kwargs.get(
            "add_symmetry_breaking", args[6] if len(args) > 6 else True
        )
        horizon = kwargs.get("horizon", args[7] if len(args) > 7 else 0)
        move_cap = kwargs.get("move_cap", args[8] if len(args) > 8 else None)
        N = kwargs.get("N", args[9] if len(args) > 9 else None)
        instance = Instance(S, L, K, orders_req, rt, p, N or {}, rt_ret=rt_return)
    else:
        rt_return = kwargs.get("rt_return", args[0] if len(args) > 0 else None)
        add_symmetry_breaking = kwargs.get(
            "add_symmetry_breaking", args[1] if len(args) > 1 else True
        )
        horizon = kwargs.get("horizon", args[2] if len(args) > 2 else 0)
        move_cap = kwargs.get("move_cap", args[3] if len(args) > 3 else None)

    S, L, K, orders_req, rt, p, N = instance
    if rt_return is None:
        rt_return = instance.rt_ret
    mdl = CpoModel()
    O = sorted(orders_req.keys())
    if rt_return is None:
        rt_return = rt  # symmetric round-trip by default

    # --- demand and candidate copy counts ---
    need_count = defaultdict(int)
    for o in O:
        for k in orders_req[o]:
            need_count[k] += 1
    active_K = [k for k in K if need_count[k] > 0]
    Lcap = max(1, len(L))

    # Conservative U: one bin visit per order item requirement.
    # This prevents heuristic warmstart from failing if it doesn't cluster perfectly.
    # The CP model will prune unused F/B/R/P variables quickly anyway.
    U = {k: need_count[k] for k in K}

    # --- intervals ---
    # Station-level order window + lane window (keep I_os, I_os_lane)
    I_os, I_os_lane = {}, {}
    for o in O:
        for s in S:
            I_os[(o, s)] = mdl.interval_var(optional=True, name=f"I_os[{o},{s}]")
            for ln in L:
                I_os_lane[(o, s, ln)] = mdl.interval_var(
                    optional=True, name=f"I_os_lane[{o},{s},{ln}]"
                )

    # Consumptions: one per (order, required SKU, station)
    C = {}
    for o in O:
        for k in orders_req[o]:
            for s in S:
                C[(o, k, s)] = mdl.interval_var(
                    size=p[k], optional=True, name=f"C[{o},{k},{s}]"
                )

    # Fetch / Pick / Return / Bin-Presence, only for active SKUs
    P, F, R, B, Block = {}, {}, {}, {}, {}
    for s in S:
        for k in active_K:
            for e in range(U[k]):
                all_picks_for_this_copy = []
                for o in O:
                    if k in orders_req[o]:
                        P[(o, s, k, e)] = mdl.interval_var(
                            size=p[k], optional=True, name=f"P[{o},{s},{k},{e}]"
                        )
                        all_picks_for_this_copy.append(P[(o, s, k, e)])
                F[(s, k, e)] = mdl.interval_var(
                    size=rt[k], optional=True, name=f"F[{s},{k},{e}]"
                )
                R[(s, k, e)] = mdl.interval_var(
                    size=rt_return[k], optional=True, name=f"R[{s},{k},{e}]"
                )
                B[(s, k, e)] = mdl.interval_var(
                    optional=True, name=f"B[{s},{k},{e}]"
                )  # free size
                # Presence coupling
                mdl.add(mdl.presence_of(B[(s, k, e)]) == mdl.presence_of(F[(s, k, e)]))
                mdl.add(mdl.presence_of(R[(s, k, e)]) == mdl.presence_of(B[(s, k, e)]))

                # The bin arrival (B) is present IFF at least one order's pick (P) uses it.
                # This links the presence of P[(o,s,k,e)] to B[(s,k,e)].
                if all_picks_for_this_copy:
                    # any_P_present is a 0/1 expression that is 1 if any P is present
                    any_P_present = mdl.max(
                        mdl.presence_of(iv) for iv in all_picks_for_this_copy
                    )
                    mdl.add(mdl.presence_of(B[(s, k, e)]) == any_P_present)
                else:
                    # No orders exist, so this B should never be present
                    mdl.add(mdl.presence_of(B[(s, k, e)]) == 0)

                # Temporal links: F -> P -> R
                for o in O:
                    if k in orders_req[o]:
                        mdl.add(mdl.end_before_start(F[(s, k, e)], P[(o, s, k, e)]))
                        mdl.add(
                            mdl.end_before_start(P[(o, s, k, e)], R[(s, k, e)])
                        )  # start(R) >= end(P)
                    # mdl.add(mdl.end_before_end(P[(o, s, k, e)], B[(s, k, e)])) does not help
                    # mdl.add(mdl.start_before_start(B[(s, k, e)], P[(o, s, k, e)]))

                # Bin presence window: [end(F), start(R)]
                mdl.add(mdl.start_at_end(B[(s, k, e)], F[(s, k, e)]))
                mdl.add(mdl.end_at_start(B[(s, k, e)], R[(s, k, e)]))

                # This Block spans from the start of Fetch to the end of Return
                Block[(s, k, e)] = mdl.interval_var(
                    optional=True, name=f"Block[{s},{k},{e}]"
                )
                mdl.add(mdl.span(Block[(s, k, e)], [F[(s, k, e)], R[(s, k, e)]]))

    if horizon == 0:
        horizon = sum((rt[k] + p[k] + rt_return.get(k, rt[k])) * U[k] for k in active_K)
        print(f"New horizon {horizon}")
    if move_cap is not None:
        moves = 0
        for s in S:
            for k in active_K:
                for e in range(U[k]):
                    moves += mdl.pulse(F[(s, k, e)], 1)
                    moves += mdl.pulse(R[(s, k, e)], 1)

        mdl.add(mdl.always_in(moves, (0, horizon), 0, move_cap))

    # --- assignment & lanes ---
    # (i) an order chooses exactly ONE station (via I_os presence)
    for o in O:
        mdl.add(mdl.sum(mdl.presence_of(I_os[(o, s)]) for s in S) == 1)

    # (ii) at chosen station, I_os equals exactly one lane window
    for o in O:
        for s in S:
            mdl.add(mdl.alternative(I_os[(o, s)], [I_os_lane[(o, s, ln)] for ln in L]))

    # (iii) lanes are unary (capacity L per station)
    for s in S:
        for ln in L:
            lane_set = [I_os_lane[(o, s, ln)] for o in O]
            if len(lane_set) >= 2:
                mdl.add(mdl.no_overlap(lane_set))

    # --- order completion = all consumptions at chosen station ---
    for o in O:
        R_o = [k for k in orders_req[o]]
        for s in S:
            if R_o:
                mdl.add(mdl.span(I_os[(o, s)], [C[(o, k, s)] for k in R_o]))
            else:
                mdl.add(mdl.length_of(I_os[(o, s)]) == 0)
            for k in R_o:
                mdl.add(mdl.presence_of(C[(o, k, s)]) == mdl.presence_of(I_os[(o, s)]))

    # --- bind each consumption to one pick at same station ---
    for o in O:
        for k in orders_req[o]:
            for s in S:
                Uk = U[k]
                if Uk <= 0:
                    # no demand => no picks exist; but we only create C for required k, so Uk>0 here
                    mdl.add(mdl.presence_of(C[(o, k, s)]) == 0)
                else:
                    candidates = [P[(o, s, k, e)] for e in range(Uk)]
                    mdl.add(mdl.alternative(C[(o, k, s)], candidates))

    # --- capacities (disjunctive only) ---
    # (1) Exactly one bin present at any station s at any time
    for s in S:
        bins_here = [B[(s, k, e)] for k in active_K for e in range(U[k])]
        if len(bins_here) >= 2:
            mdl.add(mdl.no_overlap(bins_here))

    # (2) Physical-bin capacity per SKU globally (<= Q[k] concurrent Blocks)
    #     - If Q[k] == 1: keep strong no_overlap propagation (v4 behavior)
    #     - Else: cumulative cap via step function pulses over Block intervals
    for k in active_K:
        family = [Block[(s, k, e)] for s in S for e in range(U[k])]
        if len(family) <= 1:
            continue

        if N[k] <= 1 and len(family) >= 2:
            mdl.add(mdl.no_overlap(family))
        if N[k] > len(family):
            # 1) Can't overlap more than the number of intervals you created
            continue
        if move_cap is not None and N[k] >= (len(S) + int(move_cap)):
            # 2) If move_cap exists, then at most move_cap bins can be moving (F/R) globally at any time,
            #    plus at most |S| bins can be at stations (B stage). So overlap for any single SKU
            #    can't exceed |S| + move_cap.
            continue

        bin_usage = 0
        for s in S:
            for e in range(U[k]):
                # pulse(interval, amount) adds 1 to the function during the Block
                bin_usage += mdl.pulse(Block[(s, k, e)], 1)

                # Constrain maximum concurrent usage to the available bins N[k]
        mdl.add(mdl.always_in(bin_usage, (0, horizon), 0, N[k]))

    # --- symmetry breaking ---
    if add_symmetry_breaking:
        # print("Adding symmetry breaking constraints...") # Quieter for benchmark
        # (A) Lane fill order: usage(L0) >= usage(L1) >= ... per station
        for s in S:
            for i in range(len(L) - 1):
                mdl.add(
                    mdl.sum(mdl.presence_of(I_os_lane[(o, s, i)]) for o in O)
                    >= mdl.sum(mdl.presence_of(I_os_lane[(o, s, i + 1)]) for o in O)
                )

        # (B) Ordered pick copies: for each (s,k), present copies form a prefix and are chained
        for s in S:
            for k in active_K:
                Uk = U[k]
                for e in range(Uk - 1):
                    # if P_{e+1} is present => P_e must be present  (prefix)
                    # mdl.add(mdl.if_then(mdl.presence_of(P[(s, k, e + 1)]) == 1,
                    #                     mdl.presence_of(P[(s, k, e)]) == 1))
                    mdl.add(
                        mdl.if_then(
                            mdl.presence_of(B[(s, k, e + 1)]) == 1,
                            mdl.presence_of(B[(s, k, e)]) == 1,
                        )
                    )
                    # and order them in time
                    # mdl.add(mdl.end_before_start(P[(s, k, e)], P[(s, k, e + 1)]))
                    mdl.add(mdl.end_before_start(B[(s, k, e)], B[(s, k, e + 1)]))

        # (C) Orders assigned in order to stations
        # Station load: number of assigned orders at station s
        # load = {}
        # for s in S:
        #     load[s] = mdl.sum(mdl.presence_of(I_os[o, s]) for o in O)
        #
        # # Symmetry breaking: non-increasing loads by station index
        # for i in range(len(S) - 1):
        #     mdl.add(load[S[i]] >= load[S[i+1]])

    # --- maximal horizon ---
    if horizon > 0:
        # print(f"Adding maximal horizon constraint: end <= {horizon}") # Quieter
        for o in O:
            for s in S:
                # This constrains the end time *if* the interval is present
                mdl.add(mdl.end_of(I_os[(o, s)]) <= horizon)

    # --- objective (makespan over station windows) ---
    per_order_end = [mdl.max([mdl.end_of(I_os[(o, s)]) for s in S]) for o in O]
    mdl.minimize(mdl.max(per_order_end))

    handles = {
        "I_os_lane": I_os_lane,
        "I_os": I_os,
        "C": C,
        "P": P,
        "F": F,
        "R": R,
        "B": B,
        "Block": Block,
        "U": U,
        "orders_req": orders_req,
        "rt": rt,
        "rt_return": rt_return,
        "p": p,
        "S": S,
        "L": L,
        "K": K,
        "O": O,
        "N": N,
        "move_cap": move_cap,
    }
    return mdl, handles

In [29]:
config = {
    'stations': 2, 'lanes': 4, 'orders': 20, 'pick': 1,
    'timelimit': 200, 'symmetry_breaking': True, 'skus': 10000,
    'movecap': 30, 'seed': 42, 'verbose': True, 'collect_progress': True,
    'horizon': 10000, 'alpha': 1.0, 'beta': 1.0
}
instance = generate_data(
    num_stations=config['stations'],
    lanes_per_station=config['lanes'],
    num_orders=config['orders'],
    num_skus=config['skus'],
    seed=config['seed'],
    movecap=config['movecap']
)


In [ ]:
from autostore_heuristic import validate_solution


def validate_print(heur_sol, instance, config):
    violations = validate_solution(
            heur_sol, instance,
            horizon=config['horizon'], move_cap=config['movecap']
        )
    
    if violations:
        print(f"VALIDATION FAILED ({len(violations)} violations)")
        for v in violations[:10]:
            print(f"  Violation: {v}")
    else:
        print("Validation PASSED")

In [ ]:

import time

from heuristic_rdi_sgc import run_rdi_sgc

print("Running RDI-SGC Heuristic...")
t_heur = time.perf_counter()
heur_sol = run_rdi_sgc(
    instance,
    horizon=config['horizon'], move_cap=config['movecap'], ALPHA=config['alpha'], BETA=config['beta']
)

print(f"\n=== RDI-SGC Heuristic Result ===")
print(f"Feasible:    {heur_sol.feasible}")
print(f"Makespan:    {heur_sol.makespan}")
print(f"Total bin events (moves/2): {heur_sol.total_moves // 2}")
print(f"Time:        {time.perf_counter() - t_heur:.4f}s")

validate_print(heur_sol, instance, config)

Running RDI-SGC Heuristic...

=== RDI-SGC Heuristic Result ===
Feasible:    True
Makespan:    207
Total bin events (moves/2): 78
Time:        0.1369s
Validation PASSED


In [ ]:
from cp_model import inject_warmstart, validate_warmstart

mdl, handles = build_model(
        instance, rt_return=instance.rt_ret,
        add_symmetry_breaking=config['symmetry_breaking'],
        horizon=config['horizon'], move_cap=config['movecap']
    )


mdl.set_starting_point(sp)
print("Successfully injected starting point.")

Warmstart Violations Found (2):
 - Symmetry (A) raw lanes at station 0: lane 1 has 2 orders < lane 2 has 3 (inject_warmstart will remap)
 - Symmetry (A) raw lanes at station 1: lane 0 has 1 orders < lane 1 has 3 (inject_warmstart will remap)
Successfully injected starting point.


In [33]:
sp = inject_warmstart(heur_sol, heur_sol.pick_events, mdl, handles)
mdl.set_starting_point(sp)
print("Successfully injected starting point.")

print(f"Solving CP Model with {config['timelimit']}s time limit...")
sol_cp = mdl.solve(
    TimeLimit=10,
    LogVerbosity="Terse"
)

Successfully injected starting point.
Solving CP Model with 200s time limit...
 ! --------------------------------------------------- CP Optimizer 22.2.0.0 --
 ! Minimization problem - 1250 variables, 1985 constraints
 ! Presolve      : 632 extractables eliminated
 ! Using starting point solution
 ! TimeLimit            = 10
 ! LogVerbosity         = Terse
 ! Initial process time : 0.00s (0.00s extraction + 0.00s propagation)
 !  . Log search space  : 7310.4 (before), 7310.4 (after)
 !  . Memory usage      : 6.8 MB (before), 6.8 MB (after)
 ! Using parallel search with 10 workers.
 ! ----------------------------------------------------------------------------
 !          Best Branches  Non-fixed    W       Branch decision
                        0       1250                 -
 + New bound is 0
 ! Using iterative diving.
 ! Starting point is complete and consistent with constraints.
 *           207        1  0.08s        1      (gap is 100.0%)
 *           190      490  0.08s        1 

Iter 0: Solving Partial CP Model
Optimizing 312
 ! --------------------------------------------------- CP Optimizer 22.2.0.0 --
 ! Minimization problem - 2482 variables, 4233 constraints
 ! Presolve      : 632 extractables eliminated
 ! Using starting point solution
 ! FailLimit            = 250000
 ! LogVerbosity         = Terse
 ! Initial process time : 0.01s (0.01s extraction + 0.00s propagation)
 !  . Log search space  : 7980.1 (before), 7980.1 (after)
 !  . Memory usage      : 8.0 MB (before), 8.0 MB (after)
 ! Using parallel search with 10 workers.
 ! ----------------------------------------------------------------------------
 !          Best Branches  Non-fixed    W       Branch decision
                        0       2482                 -
 + New bound is 0
 ! Using iterative diving.
 ! Giving up trying to complete the starting point to a full assignment.
 *           234     7637  0.40s        2      (gap is 100.0%)
 *           164     8186  0.42s        2      (gap is 100.

CpoException: Argument 'expr' should be a CpoExpr or an iterable of CpoExpr